# 01-reach (part 1): a two-wheeler OSRM graph and a few routes

How long a rider takes between two points. OSRM (the open-source routing engine) runs in Docker on the clipped OSM
extract with a two-wheeler profile, and answers **free-flow** routes: no traffic, no queues. Later, in this notebook,
**travel time = free-flow time × congestion multiplier** (a lever, 1.0 to 2.0); here only the free-flow part exists.

This part builds the graph on the 8 GB machine, records peak memory and time per build step, and queries a handful of
routes on the slice as a sanity check (not the stage 1 spot-check). Candidate sites and the travel-time table
follow in the second part.

In [ ]:
import json
import re
import subprocess
import sys
import os
import urllib.request
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Geod
from shapely.geometry import LineString, Point, box

from catchment.area import load_area
from catchment.fetch import DATA

ROOT = DATA.parent
NB = ROOT / "notebooks"
OSRM_DIR = DATA.resolve() / "osrm"
OSRM = "http://localhost:5001"
IMAGE = "ghcr.io/project-osrm/osrm-backend:v26.10.0-debian"   # same pin as notebooks/osrm.sh
FIGS = NB / "figures"
FIGS.mkdir(parents=True, exist_ok=True)
area = load_area()

## Assumptions

Every edit to the profile `notebooks/two-wheeler.lua`, a copy of OSRM's `car.lua` from the pinned image, in one
place; each is commented in the file with `-- two-wheeler:`. OSRM ships no two-wheeler profile and no maintained
community one exists. **Verified** means checked against the source on the date shown; **Assumption** means a
labelled choice with no external source. Anything not listed is as in `car.lua`.

In [ ]:
def lua_speeds(text):
    """highway class -> km/h from the `speeds = Sequence { highway = {...} }` block of an OSRM profile."""
    block = text[text.index("speeds = Sequence"):text.index("service_penalties")]
    return {k: int(v) for k, v in re.findall(r"^\s*(\w+)\s*=\s*(\d+)\s*,?\s*(?:--.*)?$", block, flags=re.M)}

car_lua = subprocess.run(["docker", "run", "--rm", IMAGE, "cat", "/opt/car.lua"], check=True, capture_output=True, text=True).stdout
two_lua = (NB / "two-wheeler.lua").read_text()
car_speeds, two_speeds = lua_speeds(car_lua), lua_speeds(two_lua)
speed_table = pd.DataFrame({"car.lua (km/h)": car_speeds, "two-wheeler.lua (km/h)": two_speeds})
speed_table.index.name = "highway class"
assert list(speed_table.index) == list(car_speeds) and speed_table.notna().all().all()

docker_mem = int(subprocess.run(["docker", "info", "--format", "{{.MemTotal}}"], check=True, capture_output=True, text=True).stdout)
host_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
GIB = 2**30

In [ ]:
# --- Assumptions ---------------------------------------------------------
assumptions = pd.DataFrame(
    [
        ("Road-class speeds", "Lower than car.lua in every class: " + ", ".join(f"{k} {speed_table.loc[k, 'car.lua (km/h)']} to {speed_table.loc[k, 'two-wheeler.lua (km/h)']}" for k in ["motorway", "primary", "residential", "service"]) + " km/h (table below)", "No external source; a delivery rider's free-flow speed, guessed. Gurugram's real speeds are mostly set by traffic, which the congestion multiplier stands in for (decision log, 2026-10-29)", "-", "Assumption"),
        ("Motorcycle access", "`motorcycle` is the most specific access tag (then motor_vehicle, vehicle, access) and replaces `motorcar`, so motorcycle=no / yes decide access; also read for oneway:motorcycle and for restriction and except=motorcycle on turn restrictions", "OSM access-tag hierarchy, https://wiki.openstreetmap.org/wiki/Key:access. The extract has about 250 ways with a motorcycle tag, mostly motorcycle=no (counted with osmium on the clipped extract). Whether the expressway carriageways ban two-wheelers is not checked; where unmapped, a two-wheeler may use any road a car can", "2026-10-29", "Unverified"),
        ("maxspeed tags", "Can only lower the class speed, never raise it (still × speed_reduction 0.8, as in car.lua)", "In car.lua a maxspeed tag overwrites the class speed, so a tagged 80 km/h on a residential road would beat the two-wheeler class speed; the profile wraps the handler to cap it. The extract has maxspeed values up to 120. Design choice, not a fact", "-", "Assumption"),
        ("Dimensions, penalties, turns", "Unchanged: vehicle size limits, narrow-road and service-road penalties, turn penalties, barriers, u-turn penalty 20 s", "car.lua kept as is; a rider fits where a car does and probably filters through more, so some times may be high", "-", "Assumption"),
        ("Algorithm", "Contraction hierarchies (osrm-extract, osrm-contract, osrm-routed --algorithm ch)", "Fastest many-to-many tables, which the travel-time table needs; MLD would need osrm-partition and osrm-customize", "-", "Assumption"),
        ("OSRM image", IMAGE, "https://github.com/Project-OSRM/osrm-backend/pkgs/container/osrm-backend, linux/arm64 build (plain v26.10.0 tag does not exist)", "2026-10-29", "Verified"),
        ("Docker Desktop memory limit", f"{docker_mem / GIB:.2f} GiB ({docker_mem:,} bytes) of {host_ram / GIB:.0f} GiB host RAM", "`docker info --format '{{.MemTotal}}'` and `os.sysconf`, measured when this notebook runs", "2026-10-29", "Verified"),
    ],
    columns=["Input", "Value", "Source", "Date", "Status"],
)
pd.set_option("display.max_colwidth", None)
display(assumptions.style.hide(axis="index").set_properties(**{"text-align": "left"}))
display(speed_table)

The speeds are read from the two Lua files, not copied by hand. They are free-flow, so a car's 90 km/h on a
motorway becomes 60 for a rider: roughly a loaded delivery bike on an open road. On a service road or in a
living street the rider crawls. These values are guesses to be revisited when the stage 1 spot-check shows how far
the free-flow times are from a map app's.

## Build

`notebooks/osrm.sh build` runs `osrm-extract` then `osrm-contract` on `data/osm.osm.pbf` (the extract, clipped to the
area plus 5 km), each in its own container. The script records each step's peak memory (the container's cgroup
`memory.peak`) and wall time in `data/osrm/build-stats.tsv`, and the step logs next to it. The graph stays in
`data/osrm/` and is not committed. It rebuilds each time, about two minutes.

In [ ]:
build = subprocess.run([str(NB / "osrm.sh"), "build"], capture_output=True, text=True)
if build.returncode:
    print(build.stderr)
    build.check_returncode()

stats = pd.read_csv(OSRM_DIR / "build-stats.tsv", sep="\t")
stats["peak MB"] = (stats.peak_bytes / 1e6).round(0)
stats["seconds"] = stats.seconds.round(1)
display(stats[["step", "peak MB", "seconds"]].style.hide(axis="index"))

pbf_mb = (DATA / "osm.osm.pbf").stat().st_size / 1e6
extract_log = (OSRM_DIR / "extract.log").read_text()
contract_log = (OSRM_DIR / "contract.log").read_text()
raw = re.search(r"Raw input contains (\d+) nodes, (\d+) ways, and (\d+) relations, (\d+) restrictions", extract_log).groups()
edges = int(re.search(r"Contracted graph has (\d+) edges", contract_log).group(1))
problems = [l for l in (extract_log + contract_log).splitlines() if re.search(r"\[(warning|error)\]", l, re.I)]
print(f"Input pbf: {pbf_mb:.1f} MB; {int(raw[0]):,} nodes, {int(raw[1]):,} ways, {int(raw[2]):,} relations, {int(raw[3])} turn restrictions")
print(f"Contracted graph: {edges:,} edges; graph files in data/osrm/: {sum(f.stat().st_size for f in OSRM_DIR.glob('osm.osrm*')) / 1e6:.0f} MB")
print(f"Warnings or errors in the step logs: {len(problems)}")
for l in problems[:10]:
    print(" ", l)
print(f"Peak memory vs the Docker limit: {stats.peak_bytes.max() / docker_mem:.0%}")

## Serve and query routes

`osrm-routed` runs as the container `catchment-osrm` on `localhost:5001` (macOS often holds 5000). The slice is the
same one `00-demand` uses: a lon/lat box cut to the area. Seven routes inside it, short to long, between old
Gurugram (sectors 4-14, Sadar) and the Golf Course Road / Sohna Road side. The endpoints are coordinates of Overture
places with the named area in their name; "near" means the coordinates are one such place's, not a surveyed landmark.

In [ ]:
subprocess.run([str(NB / "osrm.sh"), "serve"], check=True)

slice_poly = box(77.005, 28.405, 77.105, 28.480).intersection(area)

# short name -> (lon, lat); every point is the position of an Overture place named for the area
POINTS = {
    "Sadar Bazar": (77.0293, 28.4612),
    "Civil Lines": (77.0476, 28.4630),
    "Rajiv Chowk": (77.0322, 28.4501),
    "Medanta": (77.0569, 28.4712),
    "Sikanderpur": (77.1026, 28.4752),
    "Sector 56": (77.0983, 28.4250),
    "Sohna Rd, sec 48": (77.0400, 28.4114),
    "Golf Course Rd, sec 57": (77.0740, 28.4151),
}
for name, (x, y) in POINTS.items():
    assert slice_poly.contains(Point(x, y)), f"{name} is outside the slice"

ROUTES = [  # (from, to)
    ("Sadar Bazar", "Civil Lines"),
    ("Rajiv Chowk", "Sadar Bazar"),
    ("Sadar Bazar", "Medanta"),
    ("Sohna Rd, sec 48", "Golf Course Rd, sec 57"),
    ("Medanta", "Sikanderpur"),
    ("Sadar Bazar", "Sector 56"),
    ("Sohna Rd, sec 48", "Sikanderpur"),
]

In [ ]:
geod = Geod(ellps="WGS84")

def route(a, b):
    (x1, y1), (x2, y2) = POINTS[a], POINTS[b]
    url = f"{OSRM}/route/v1/driving/{x1},{y1};{x2},{y2}?overview=full&geometries=geojson&annotations=speed"
    with urllib.request.urlopen(url, timeout=30) as r:
        res = json.load(r)
    assert res["code"] == "Ok", res
    rt = res["routes"][0]
    _, _, straight_m = geod.inv(x1, y1, x2, y2)
    return {
        "from": a, "to": b,
        "free-flow min": rt["duration"] / 60,
        "road km": rt["distance"] / 1000,
        "straight km": straight_m / 1000,
        "detour": rt["distance"] / straight_m,
        "avg km/h": rt["distance"] / rt["duration"] * 3.6,
        "top km/h": max(rt["legs"][0]["annotation"]["speed"]) * 3.6,  # annotation speed is m/s
        "snap m": max(w["distance"] for w in res["waypoints"]),
        "geometry": LineString(rt["geometry"]["coordinates"]),
    }

routes = pd.DataFrame([route(a, b) for a, b in ROUTES])
routes.index = range(1, len(routes) + 1)
routes.index.name = "route"
show = routes.drop(columns="geometry").round({"free-flow min": 1, "road km": 1, "straight km": 1, "detour": 2, "avg km/h": 1, "top km/h": 0, "snap m": 0})
display(show)

`top km/h` is the fastest annotated segment on the route (OSRM reports segment speed in m/s; converted); with the two-wheeler
profile it can never exceed the motorway class's 60. `snap m` is how far the farthest endpoint sat from the
nearest road.

In [ ]:
bad_speed = routes[(routes["avg km/h"] < 15) | (routes["avg km/h"] > 35)]
bad_detour = routes[(routes.detour < 1.2) | (routes.detour > 1.8)]
bad_snap = routes[routes["snap m"] > 100]
print("Plausible: average 15-35 km/h, detour ratio 1.2-1.8, endpoints within 100 m of a road.")
print("Average speed outside 15-35 km/h:", list(bad_speed.index) or "none")
print("Detour ratio outside 1.2-1.8:", list(bad_detour.index) or "none")
print("Endpoint more than 100 m from a road:", list(bad_snap.index) or "none")
print(f"Top segment speed across routes: {routes['top km/h'].max():.0f} km/h (class cap 60)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
gpd.GeoSeries([slice_poly], crs=4326).plot(ax=ax, facecolor="#f2f2f2", edgecolor="black", linewidth=1)
colors = plt.cm.tab10(np.arange(len(routes)))
for (i, r), c in zip(routes.iterrows(), colors):
    x, y = r.geometry.xy
    ax.plot(x, y, color=c, linewidth=2.5, alpha=0.9,
            label=f"{i}  {r['from']} to {r['to']}: {r['free-flow min']:.1f} min, {r['road km']:.1f} km")
    mid = r.geometry.interpolate(0.5, normalized=True)
    ax.annotate(f"{r['free-flow min']:.0f}", (mid.x, mid.y), ha="center", va="center", fontsize=9, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=c, lw=1.5))
for name, (x, y) in POINTS.items():
    ax.plot(x, y, "o", color="black", markersize=6, zorder=5)
    ax.annotate(name, (x, y), xytext=(5, 5), textcoords="offset points", fontsize=8, zorder=6)
ax.set_aspect(1 / np.cos(np.radians(28.44)))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, ncol=1, frameon=False)
ax.set_title("Free-flow two-wheeler routes on the slice (labels are minutes)")
fig.savefig(FIGS / "01-routes.png", dpi=120, bbox_inches="tight")
plt.show()

![Free-flow two-wheeler routes on the slice](figures/01-routes.png)

In [ ]:
_ = subprocess.run([str(NB / "osrm.sh"), "stop"], check=True)

Results from the run on 2026-10-29 (copied here because outputs are stripped on commit; re-running updates the
printed tables, not this one):

Docker Desktop limit 3.82 GiB (4,106,604,544 bytes) on an 8 GiB host, image `ghcr.io/project-osrm/osrm-backend:v26.10.0-debian`
(arm64), CH algorithm. Input `data/osm.osm.pbf` 16.6 MB: 1,913,652 nodes, 358,163 ways, 694 relations, 102 turn
restrictions. No warnings or errors in the step logs.

| step | peak memory (MB) | wall time (s) |
|---|---:|---:|
| `osrm-extract` | 491 | 23.8 |
| `osrm-contract` | 213 | 123.1 |

The peak is 12% of Docker's limit, so the clipped extract builds with plenty of room. The graph (1.81 M contracted
edges) takes 117 MB on disk. Other builds the same day took 16-24 s and 101-123 s: wall time moves with machine load,
peak memory barely does. Whether the full area fits is a stage 1 question; this extract is already the area plus a
5 km buffer.

| route | from | to | free-flow min | road km | straight km | detour | avg km/h |
|---:|---|---|---:|---:|---:|---:|---:|
| 1 | Sadar Bazar | Civil Lines | 7.2 | 2.9 | 1.8 | 1.59 | 24.0 |
| 2 | Rajiv Chowk | Sadar Bazar | 5.4 | 2.5 | 1.3 | 2.02 | 28.6 |
| 3 | Sadar Bazar | Medanta | 7.7 | 3.8 | 2.9 | 1.30 | 29.8 |
| 4 | Sohna Rd, sec 48 | Golf Course Rd, sec 57 | 10.0 | 5.3 | 3.4 | 1.57 | 31.5 |
| 5 | Medanta | Sikanderpur | 13.1 | 6.9 | 4.5 | 1.54 | 31.6 |
| 6 | Sadar Bazar | Sector 56 | 20.3 | 9.2 | 7.9 | 1.17 | 27.3 |
| 7 | Sohna Rd, sec 48 | Sikanderpur | 23.8 | 12.5 | 9.4 | 1.34 | 31.5 |

Sanity check: every average speed is inside 15-35 km/h and every endpoint within 50 m of a road. Two detour ratios
fall outside 1.2-1.8: route 2 (2.02) and route 6 (1.17). Route 2 is short (1.3 km straight), and its line runs past
the endpoint before doubling back, which looks like a destination on the far carriageway of a divided road; not
checked against OSM. Route 6 is nearly straight, which is allowed. OSRM's per-segment speeds
cluster at the two-wheeler class values (20, 30, 35, 40, 50 km/h), not car.lua's (25, 40, 55, 65, 85), so the profile
edits took effect. Free-flow times are model output, not checked against a map app yet (stage 1
spot-check). They already include turn penalties, so the averages sit below the class speeds.

## Next

Part 2 (#4) starts the server again with `notebooks/osrm.sh serve`, adds candidate sites, and builds the travel-time
table of candidate site × hex pairs within 17 min free-flow.